# Bet Sizing

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.backtesting.backtest_statistics import compute_strategy_returns
from src.backtesting.portfolio import (
    simulate_portfolio,
    validate_event_prices,
)
from src.backtesting.bet_sizing import (
    build_target_positions,
    bet_size,
    get_signal,
    get_target_position,
    get_w,
    inverse_price,
    limit_price,
)

period = "2025-01-01_2025-12-31"
artifact_dir = PROJECT_ROOT / "data/model_artifact"
event_dir = PROJECT_ROOT / "data/research_data/events"
result_dir = PROJECT_ROOT / "data/backtest_results"
result_dir.mkdir(parents=True, exist_ok=True)

prediction_path = artifact_dir / "meta_predictions.parquet"
event_path = event_dir / f"aapl_model_events_{period}.parquet"
raw_price_path = (
    PROJECT_ROOT / "data/research_data/market/data"
    / f"aapl_{period}.parquet"
)
result_path = result_dir / "event_strategy_returns.parquet"

predictions = pd.read_parquet(prediction_path).sort_index()
num_classes = 2
step_size = 0.10


## Final Signed Target Positions

- **Purpose:** Generate the final holdout target-position series once in Bet Sizing and pass it directly to portfolio accounting.
- **Settings:** Average primary sides without discretization. For meta signals, `get_signal` applies direction before averaging and discretizes at `step_size=0.10`; zero signals remain in the mean.
- **Data:** Use holdout predictions only, with all event-start and event-end boundaries.
- **Decision:** Signal generation belongs to `bet_sizing.build_target_positions`; portfolio accounting consumes the result. Equivalence checks for the sizing interfaces live in tests.


In [ ]:
target_positions = build_target_positions(predictions, step_size=step_size)
display(target_positions)


## Legacy Event Analysis Artifact

- **Purpose:** Preserve `event_strategy_returns.parquet` for synthetic-backtest consumers and event classification inputs.
- **Settings:** Use the established unsigned sizing path through `get_signal`, then apply event direction in `compute_strategy_returns`; preserve both development and holdout rows.
- **Decision:** These event returns are compatibility outputs, not inputs to current portfolio investment performance. They are distinct from signed targets aggregated before accounting.


In [ ]:
event_windows = predictions[["event_end"]].rename(
    columns={"event_end": "t1"}
)
probability_bet_sizes = get_signal(
    events=event_windows,
    step_size=step_size,
    prob=predictions["meta_probability"],
    pred=predictions["meta_action"],
    num_classes=num_classes,
)

predictions = predictions.copy()
predictions["bet_size"] = probability_bet_sizes.reindex(predictions.index)
if predictions["bet_size"].isna().any():
    raise ValueError("Every prediction must have an event-start bet size.")
if not predictions["bet_size"].between(0.0, 1.0).all():
    raise ValueError("Event-start bet sizes must be in [0, 1].")

strategy_returns = compute_strategy_returns(
    predictions,
    one_way_cost_bps=0.0,
)
strategy_returns.to_parquet(result_path)

summary = strategy_returns.groupby("partition").agg(
    events=("raw_return", "size"),
    meta_acted=("meta_action", "sum"),
    average_bet_size=("bet_size", "mean"),
    primary_net_return=("primary_only_net_return", "sum"),
    meta_net_return=("meta_filtered_net_return", "sum"),
    meta_total_cost=("meta_filtered_total_cost", "sum"),
)
display(summary)
display(strategy_returns.head())
print(result_path)

## Self-Financing Holdout Portfolios

- **Purpose:** Convert signed active signals into one account per strategy and persist the account ledger for all investment statistics.
- **Settings:** Start each holdout account at 100,000 USD; rebalance at every event boundary; reinvest gains and losses; use zero bps per side and fractional shares.
- **Data:** Use exact raw AAPL trade prices, keeping the last observation at duplicate timestamps; require boundary returns to reproduce the saved event returns.
- **Decision:** Follow the [portfolio accounting decisions](../../docs/decisions.md#self-financing-portfolio-evaluation). Zero meta signals remain in the active average; no development holdings carry into holdout. Event-return artifacts remain available for classification and legacy consumers.


In [ ]:
initial_aum = 100_000.0
broker_fee_bps = 1.0
slippage_bps = 1.0
prices = (
    pd.read_parquet(raw_price_path, columns=["timestamp", "price"])
    .drop_duplicates("timestamp", keep="last")
    .set_index("timestamp")["price"]
    .sort_index()
)
validate_event_prices(predictions, prices)
ledgers = {
    strategy: simulate_portfolio(
        prices,
        target_positions[strategy],
        initial_aum=initial_aum,
        broker_fee_bps=broker_fee_bps,
        slippage_bps=slippage_bps,
    )
    for strategy in target_positions
}
portfolio_ledger = pd.concat(ledgers, names=["strategy", "timestamp"])
portfolio_path = result_dir / "portfolio_ledger.parquet"
portfolio_ledger.to_parquet(portfolio_path)

display(pd.DataFrame({
    strategy: {
        "initial_aum": initial_aum,
        "final_aum": ledger["aum"].iloc[-1],
        "net_pnl": ledger["aum"].iloc[-1] - initial_aum,
        "total_traded_value": ledger["traded_value"].sum(),
        "total_broker_fee": ledger["broker_fee"].sum(),
        "total_slippage_cost": ledger["slippage_cost"].sum(),
        "total_execution_cost": ledger["execution_cost"].sum(),
    }
    for strategy, ledger in ledgers.items()
}).T)
print(portfolio_path)


## Dynamic Bet Sizes and Limit Prices

- **Purpose:** Convert point-in-time triple-barrier price forecasts into dynamic event-entry sizes, target positions, and breakeven limit prices.
- **Settings:** `pt_sl=(1.0, 1.0)`; `max_position=100`; `current_position=0`; calibrate `w` to `bet_size=0.95` at the development median absolute price divergence.
- **Data:** Join each event to its exact event-start trade price and target return in the development and holdout partitions.
- **Decision:** Fit calibration on development only, apply the frozen value to holdout, and keep these diagnostics separate from persisted probability-sized returns.

In [ ]:
event_metadata = (
    pd.read_parquet(
        event_path,
        columns=["event_start", "target_return"],
    )
    .set_index("event_start")
    .sort_index()
)
start_prices = prices.rename("market_price")
dynamic_positions = (
    predictions[["partition", "primary_side", "meta_action"]]
    .join(event_metadata, how="left", validate="one_to_one")
    .join(start_prices, how="left", validate="one_to_one")
)
if dynamic_positions[["target_return", "market_price"]].isna().any().any():
    raise ValueError("Every prediction must match an event target and start price.")

dynamic_positions["forecast_price"] = dynamic_positions["market_price"] * (
    1.0
    + dynamic_positions["primary_side"]
    * dynamic_positions["target_return"]
)
dynamic_positions["price_divergence"] = (
    dynamic_positions["forecast_price"]
    - dynamic_positions["market_price"]
)
development = dynamic_positions["partition"].eq("development")
calibration_divergence = float(
    dynamic_positions.loc[development, "price_divergence"].abs().median()
)
calibration_bet_size = 0.95
w = get_w(
    price_divergence=calibration_divergence,
    bet_size_value=calibration_bet_size,
)
max_position = 100
current_position = 0

dynamic_positions["dynamic_bet_size"] = dynamic_positions[
    "price_divergence"
].map(lambda divergence: bet_size(w, divergence))
dynamic_positions["unfiltered_target_position"] = [
    get_target_position(w, forecast_price, market_price, max_position)
    for forecast_price, market_price in zip(
        dynamic_positions["forecast_price"],
        dynamic_positions["market_price"],
        strict=True,
    )
]
dynamic_positions["target_position"] = (
    dynamic_positions["unfiltered_target_position"]
    * dynamic_positions["meta_action"]
).astype("int64")
dynamic_positions["recovered_market_price"] = [
    inverse_price(forecast_price, w, dynamic_bet_size_value)
    for forecast_price, dynamic_bet_size_value in zip(
        dynamic_positions["forecast_price"],
        dynamic_positions["dynamic_bet_size"],
        strict=True,
    )
]
dynamic_positions["limit_price"] = [
    limit_price(
        target_position=target_position,
        current_position=current_position,
        forecast_price=forecast_price,
        w=w,
        max_position=max_position,
    )
    for target_position, forecast_price in zip(
        dynamic_positions["target_position"],
        dynamic_positions["forecast_price"],
        strict=True,
    )
]

if not dynamic_positions["dynamic_bet_size"].abs().lt(1.0).all():
    raise ValueError("Dynamic bet sizes must stay inside (-1, 1).")
if not dynamic_positions["unfiltered_target_position"].abs().lt(
    max_position
).all():
    raise ValueError("Dynamic target positions must stay inside the limit.")
if not (
    np.sign(dynamic_positions["unfiltered_target_position"])
    == dynamic_positions["primary_side"]
).all():
    raise ValueError("Dynamic target directions must match primary sides.")
np.testing.assert_allclose(
    dynamic_positions["recovered_market_price"],
    dynamic_positions["market_price"],
)
passed = dynamic_positions["meta_action"].eq(0)
if not dynamic_positions.loc[passed, "target_position"].eq(0).all():
    raise ValueError("Passed events must have zero dynamic target position.")
if not dynamic_positions.loc[passed, "limit_price"].isna().all():
    raise ValueError("Passed events must not produce limit prices.")
acting = dynamic_positions["target_position"].ne(0)
lower_price = dynamic_positions[["market_price", "forecast_price"]].min(
    axis=1
)
upper_price = dynamic_positions[["market_price", "forecast_price"]].max(
    axis=1
)
if not dynamic_positions.loc[acting, "limit_price"].between(
    lower_price.loc[acting],
    upper_price.loc[acting],
).all():
    raise ValueError("Limit prices must lie between market and forecast prices.")

calibration = pd.Series(
    {
        "development_median_absolute_divergence": calibration_divergence,
        "calibration_bet_size": calibration_bet_size,
        "w": w,
        "max_position": max_position,
    },
    name="value",
)
dynamic_summary = dynamic_positions.groupby("partition").agg(
    events=("target_return", "size"),
    meta_acted=("meta_action", "sum"),
    average_absolute_bet_size=("dynamic_bet_size", lambda x: x.abs().mean()),
    average_absolute_target=("target_position", lambda x: x.abs().mean()),
)
display(calibration.to_frame())
display(dynamic_summary)
display(
    dynamic_positions[
        [
            "primary_side",
            "meta_action",
            "target_return",
            "market_price",
            "forecast_price",
            "dynamic_bet_size",
            "target_position",
            "limit_price",
        ]
    ].head()
)